# 5.5 Embedding models: BGE-small vs. Qwen3-Embedding-0.6B

The research pipeline matched claims to careers with **BGE-small** (`BAAI/bge-small-en-v1.5`, 33M parameters) until CHANGELOG 104, and with **Qwen3-Embedding-0.6B** (600M) since; the follow-up ranker uses Qwen3 too. Notebook 5.3 re-ran with Qwen3, but its ranking metrics only see candidates that have a label, and the labels were drawn from **BGE's** top 5s (CHANGELOG 031, rounds since). So here both models score the **same labelled pairs**:

- **The pool:** every labelled (career, claim) pair still eligible, 40 careers (`evaluation/claim_relevance_labels*.csv`).
- **Scores:** for each model, the career's query and the claim's shown text, as the matcher builds them; *raw* cosine and *centered* (raw minus the claim's average over all 923 careers).
- **Metrics:** within each career's labelled pool, ranked by each score: precision@1/@3, MRR, and ROC AUC (the chance a relevant pair outscores an irrelevant one of the same career).
- **Coverage:** how much of each model's real top 5 (over all claims) is labelled at all. A model whose top 5 is mostly unlabelled is judged on pairs it would not have shown, so this says how far the comparison can be trusted.

Run locally: it needs both models in the Hugging Face cache and `data/research/ai_impact_claims.jsonl` (the pipeline state). Not part of the monthly run.

In [1]:
import _setup
_setup.setup()

In [2]:
import json
import os
import sys
import warnings
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.metrics import roc_auc_score

os.environ["HF_HUB_OFFLINE"] = "1"  # both models come from the local cache
os.environ["TQDM_DISABLE"] = "1"
warnings.filterwarnings("ignore")
sys.path.insert(0, "scripts/research")
import match_claims_to_occupations as matcher  # noqa: E402
from career_rag.embeddings import embed_queries  # noqa: E402

claims = [json.loads(line) for line in matcher.CLAIMS.open(encoding="utf-8") if line.strip()]
kept = [claim for claim in claims if matcher.eligible(claim)]
shown, unshowable = matcher.shown_texts_and_unshowable(kept)
kept = [claim for claim in kept if claim["claim_id"] not in unshowable]
texts = [matcher.claim_text(claim, shown[claim["claim_id"]]) for claim in kept]
claim_index = {claim["claim_id"]: j for j, claim in enumerate(kept)}
onet = duckdb.connect(str(matcher.ONET_DUCKDB_PATH), read_only=True)
occupations = onet.execute("SELECT onetsoc_code, title, description FROM occupation_data WHERE onetsoc_code IN "
                           "(SELECT DISTINCT onetsoc_code FROM task_statements) ORDER BY onetsoc_code").fetchall()
onet.close()
career_index = {code: i for i, (code, _, _) in enumerate(occupations)}
questions = [matcher.occupation_query(title, description) for _, title, description in occupations]

labels = pd.concat(pd.read_csv(path, dtype={"relevant": int}) for path in sorted(Path("evaluation").glob("claim_relevance_labels*.csv")))
labels = labels.drop_duplicates(["onetsoc_code", "claim_id"], keep="last")
pool = labels[labels.claim_id.isin(claim_index)].copy()
print(f"claims embedded: {len(kept)}; labelled pairs: {len(labels)}, still eligible: {len(pool)}, "
      f"{pool.relevant.sum()} relevant, {pool.onetsoc_code.nunique()} careers")

claims embedded: 2483; labelled pairs: 729, still eligible: 596, 118 relevant, 40 careers


C:\Users\HP\Documents\depo\odtu\4thyear\nlp\proje\career-rag\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Both models' scores for every career and claim

In [3]:
# BGE-small, as the matcher used it until CHANGELOG 104: claims as they are, queries with BGE's prefix
bge = SentenceTransformer("BAAI/bge-small-en-v1.5", device="cpu")
bge_claims = bge.encode(texts, batch_size=64, normalize_embeddings=True)
bge_queries = bge.encode(["Represent this sentence for searching relevant passages: " + q for q in questions],
                         batch_size=64, normalize_embeddings=True)
# Qwen3: the matcher's own vectors from its latest run, by claim ID (its research_claims rows are in the vectors' order;
# checks run after it can make a few more claims unshowable), and the queries with its instruction
con = duckdb.connect(str(matcher.OUTPUT), read_only=True)
matched = [claim_id for (claim_id,) in con.execute("SELECT claim_id FROM research_claims ORDER BY rowid").fetchall()]
con.close()
qwen_all = np.load(matcher.OUTPUT.with_suffix(".claim_vectors.npy"))
assert len(qwen_all) == len(matched)
row_of = {claim_id: i for i, claim_id in enumerate(matched)}
qwen_claims = qwen_all[[row_of[claim["claim_id"]] for claim in kept]]
qwen_queries = embed_queries(questions, matcher.QUERY_INSTRUCTION)
scores = {}
for name, queries, vectors in (("BGE-small", bge_queries, bge_claims), ("Qwen3-0.6B", qwen_queries, qwen_claims)):
    raw = queries @ vectors.T
    scores[name] = {"raw": raw, "centered": raw - raw.mean(axis=0)}
rows = [career_index[code] for code in pool.onetsoc_code]
cols = [claim_index[cid] for cid in pool.claim_id]
for name in scores:
    for kind in ("raw", "centered"):
        pool[f"{name} {kind}"] = scores[name][kind][rows, cols]
pool[[c for c in pool.columns if " " in c]].describe().loc[["mean", "std", "min", "max"]].round(3)

Loading weights: 100%|██████████| 310/310 [00:00<00:00, 2900.90it/s]


,BGE-small raw,BGE-small centered,Qwen3-0.6B raw,Qwen3-0.6B centered
mean,0.617,0.051,0.448,0.072
std,0.067,0.054,0.083,0.061
min,0.381,-0.141,0.183,-0.081
max,0.784,0.193,0.685,0.337


## 2. Ranking the labelled pool, per career

Per career, then averaged: the labelled pairs sorted by each score. Only careers with at least one relevant and one irrelevant labelled pair count for AUC and MRR; precision uses all 40.

In [4]:
def metrics(column):
    out = []
    for code, group in pool.groupby("onetsoc_code"):
        ranked = group.sort_values(column, ascending=False).relevant.to_numpy()
        both = 0 < ranked.sum() < len(ranked)
        first = np.flatnonzero(ranked)
        out.append({"p@1": ranked[:1].mean(), "p@3": ranked[:3].mean(),
                     "mrr": 1 / (first[0] + 1) if len(first) else np.nan,
                     "auc": roc_auc_score(group.relevant, group[column]) if both else np.nan})
    return pd.DataFrame(out).mean().round(3)


pd.DataFrame({column: metrics(column) for column in pool.columns if " " in column})

,BGE-small raw,BGE-small centered,Qwen3-0.6B raw,Qwen3-0.6B centered
p@1,0.400,0.400,0.450,0.400
p@3,0.325,0.283,0.358,0.358
mrr,0.610,0.595,0.679,0.606
auc,0.622,0.617,0.765,0.752


## 3. How much of each model's real top 5 is labelled

In [5]:
def coverage(name, kind):
    labelled = set(zip(pool.onetsoc_code, pool.claim_id))
    found = total = relevant = 0
    for code in pool.onetsoc_code.unique():
        top, seen = [], set()
        for j in np.argsort(-scores[name][kind][career_index[code]]):
            if shown[kept[j]["claim_id"]] in seen:
                continue  # one per text, as the matcher keeps them
            seen.add(shown[kept[j]["claim_id"]])
            top.append(kept[j]["claim_id"])
            if len(top) == 5:
                break
        total += 5
        found += sum((code, cid) in labelled for cid in top)
        relevant += sum(pool[(pool.onetsoc_code == code) & (pool.claim_id.isin(top))].relevant)
    return {"top 5 labelled": f"{found}/{total}", "share labelled": round(found / total, 3),
            "relevant in top 5": relevant, "relevant per career": round(relevant / pool.onetsoc_code.nunique(), 2)}


pd.DataFrame({f"{name} {kind}": coverage(name, kind) for name in scores for kind in ("raw", "centered")}).T

,top 5 labelled,share labelled,relevant in top 5,relevant per career
BGE-small raw,175/200,0.875,50,1.25
BGE-small centered,150/200,0.75,41,1.02
Qwen3-0.6B raw,183/200,0.915,59,1.48
Qwen3-0.6B centered,185/200,0.925,67,1.68


## Findings

With the round 2026_10_qwen labels, both models' top 5s are mostly labelled (BGE 75–88%, Qwen3 92–93%), so the comparison no longer favours the model the labels came from. 596 labelled pairs are still eligible, 118 of them relevant, in 40 careers.

1. **Qwen3 ranks the labelled pairs better:** AUC 0.77 (raw) against BGE's 0.62, MRR 0.68 against 0.61, precision@1 0.45 against 0.40. Before the Qwen3 round (only BGE's top 5s labelled), the gap looked larger (AUC 0.91 against 0.72): on BGE's own pool, Qwen3 reordered pairs it had not chosen.
2. **Qwen3's top 5s hold more relevant claims:** 1.68 per career (centered) and 1.48 (raw), against BGE's 1.25 (raw) and 1.02 (centered).
3. **Centering helps neither model here** (AUC: Qwen3 0.77 raw, 0.75 centered; BGE 0.62, 0.62), but it puts more relevant claims into Qwen3's top 5 (1.68 against 1.48).
4. **The score scales differ** (raw mean 0.45 for Qwen3, 0.62 for BGE), so thresholds don't carry over between models.

**Limits:** the labels were drafted by an LLM (Claude) and accepted by you, the Qwen3 round without the scores; 40 careers is a small sample.